> **Public evidence copy:** Stored outputs and execution counts have been removed. 
> The participant-level workbook is not included. Set `JAI_SURVEY_DATA` to an 
> authorized secure copy before running this notebook.


# Roaming-dog perceptions and management

## Setup

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import os 

In [ ]:
from pathlib import Path

data_path_value = os.environ.get("JAI_SURVEY_DATA")
if not data_path_value:
    raise RuntimeError(
        "Set JAI_SURVEY_DATA to the absolute path of the restricted survey workbook. "
        "Do not copy participant data into this repository."
    )

file = Path(data_path_value).expanduser().resolve()
if not file.is_file():
    raise FileNotFoundError(f"Restricted survey workbook not found: {file}")



In [ ]:
df1 = pd.read_excel(file, 'Jai KAP Survey 2025')


In [ ]:
df1_filtered = df1[
    (df1['Is someone >18 present at the household?'] == 'Yes') &
    (df1['Has the respondent consented to undertake the survey?'] == 'Yes')
]



In [ ]:
df1_ong = df1_filtered[df1_filtered["District"] == "Ongkharak"]
df1_pak = df1_filtered[df1_filtered["District"] == "Pak Phli"]


> Records without consent or without an eligible adult respondent are excluded according to the analysis cleaning rule.


## Analysis


### Q53 – Roughly how many dogs are roaming on your soi? 


In [ ]:
# "Roughly how many dogs are roaming on your soi today?"

for name, df in [("Pak Phli", df1_pak), ("Ongkharak", df1_ong)]:
    print(f"{name}:\n{df.iloc[:, 85].value_counts().sort_index()}\n")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

columns_to_plot = [
    "Roughly how many dogs are roaming on your soi today?",
]

for col in columns_to_plot:
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Pak Phli", "Ongkharak"))
    for i, (name, df) in enumerate([("Pak Phli", df1_pak), ("Ongkharak", df1_ong)], start=1):
        counts = df[col].value_counts()
        fig.add_trace(
            go.Bar(y=counts.values, x=counts.index, name=name),
            row=1, col=i
        )
    fig.update_layout(
        height=600,
        width=1000,
        title_text=col,
        xaxis_title="Number of Dogs",
        yaxis_title="Count",
        xaxis2_title="Number of Dogs",
        yaxis2_title="Count"
    )
    fig.show()

In [ ]:
df1_ong.iloc[:,85].value_counts().sort_index()

In [ ]:
for name, df in [("Pak Phli", df1_pak), ("Ongkharak", df1_ong)]:
    print(f"{name}:\n{df.iloc[:, 85].describe()}\n")

In [ ]:
fig = px.box(
    df1_filtered,
    y="District",
    x="Roughly how many dogs are roaming on your soi today?",
    points="all",
    orientation='h',
    color="District",
    category_orders={"District": ["Pak Phli", "Ongkharak"]},
    color_discrete_map={
        "Pak Phli": "blue",
        "Ongkharak": "#d16443"   # corrected hex code
    }
)

fig.update_traces(quartilemethod="inclusive")
fig.update_layout(width=1000, height=800)
fig.show()


In [ ]:
# Apply the manuscript-documented sensitivity-analysis threshold.
df_no_outlier = df1_filtered[
    df1_filtered["Roughly how many dogs are roaming on your soi today?"] < 100
]

df1_ong_no_outlier = df1_ong[
    df1_ong["Roughly how many dogs are roaming on your soi today?"] < 100
]

In [ ]:
for name, df in [("Pak Phli", df1_pak), ("Ongkharak", df1_ong_no_outlier)]:
    print(f"{name}:\n{df.iloc[:, 85].describe()}\n")

In [ ]:
fig = px.box(
    df_no_outlier,
    y="District",
    x="Roughly how many dogs are roaming on your soi today?",
    points="all",
    orientation='h',
    color="District",
    category_orders={"District": ["Pak Phli", "Ongkharak"]},
    color_discrete_map={
        "Pak Phli": "blue",
        "Ongkharak": "#d16443"   # corrected hex code
    }
)

fig.update_traces(quartilemethod="inclusive")
fig.update_layout(width=1000, height=800)
fig.show()


### Q54 – What do you think about the number of dogs on your soi at present? 

In [ ]:
for name, df in [("Pak Phli", df1_pak), ("Ongkharak", df1_ong)]:
    print(f"{name}:\n{df.iloc[:, 86].value_counts().sort_index()}\n")

In [ ]:
columns_to_plot = [
    "What do you think about the number of dogs on your soi?",
]

for col in columns_to_plot:
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Pak Phli", "Ongkharak"))
    for i, (name, df) in enumerate([("Pak Phli", df1_pak), ("Ongkharak", df1_ong)], start=1):
        counts = df[col].value_counts()
        fig.add_trace(
            go.Bar(y=counts.values, x=counts.index, name=name),
            row=1, col=i
        )
    fig.update_layout(
        height=600,
        width=1400,
        title_text=col,
        xaxis_title="Opinions",
        yaxis_title="Count",
        xaxis2_title="Opinions",
        yaxis2_title="Count"
    )
    fig.show()

In [ ]:
columns_to_plot = [
    "What do you think about the number of dogs on your soi?",
]

cat_order = [
    "I am unhappy",
    "I do not accept",
    "Neutral",
    "I am accepting/OK",
    "I am happy"
]

for col in columns_to_plot:
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Pak Phli", "Ongkharak"))

    for i, (name, df) in enumerate([("Pak Phli", df1_pak), ("Ongkharak", df1_ong)], start=1):
        
        # Count responses, reindex with the correct order
        counts = df[col].value_counts().reindex(cat_order, fill_value=0)

        fig.add_trace(
            go.Bar(
                y=counts.values,
                x=counts.index,
                name=name
            ),
            row=1, col=i
        )

    fig.update_layout(
        height=600,
        width=1400,
        title_text=col,
        xaxis_title="Opinions",
        yaxis_title="Count",
        xaxis2_title="Opinions",
        yaxis2_title="Count"
    )

    fig.show()


In [ ]:
col = "What do you think about the number of dogs on your soi?"

cat_order = [
    "I am unhappy",
    "I do not accept",
    "Neutral",           # keep as-is if that's how it's spelled in your data
    "I am accepting/OK",
    "I am happy"
]

neg_cats = ["I am unhappy", "I do not accept"]
neutral_cat = "Netural"
pos_cats = ["I am accepting/OK", "I am happy"]

color_map = {
    "I am unhappy": "#b30000",          # dark red
    "I do not accept": "#e34a33",       # lighter red
    "Neutral": "#c5c4c4",               # white/very light
    "I am accepting/OK": "#74add1",     # light blue
    "I am happy": "#4575b4"             # darker blue
}


In [ ]:
def prep_percent(df, col, cat_order):
    counts = df[col].value_counts().reindex(cat_order, fill_value=0)
    perc = counts / counts.sum() * 100
    return perc


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# get % distribution for each district
pak = prep_percent(df1_pak, col, cat_order)
ong = prep_percent(df1_ong, col, cat_order)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Pak Phli", "Ongkharak"),
    shared_yaxes=True
)

# ---- Pak Phli ----
x_pak = []
colors_pak = []
for c in cat_order:
    val = pak[c]
    if c in neg_cats:
        val = -val  # send negatives to the left
    x_pak.append(val)
    colors_pak.append(color_map[c])

fig.add_trace(
    go.Bar(
        y=cat_order,
        x=x_pak,
        orientation="h",
        marker_color=colors_pak,
        showlegend=False
    ),
    row=1, col=1
)

# ---- Ongkharak ----
x_ong = []
colors_ong = []
for c in cat_order:
    val = ong[c]
    if c in neg_cats:
        val = -val
    x_ong.append(val)
    colors_ong.append(color_map[c])

fig.add_trace(
    go.Bar(
        y=cat_order,
        x=x_ong,
        orientation="h",
        marker_color=colors_ong,
        showlegend=False
    ),
    row=1, col=2
)

# layout
fig.update_layout(
    barmode="relative",
    title="Q54 – Perception of Dog Numbers on the Soi (Diverging Sentiment by District)",
    height=650,
    width=1400,
    xaxis_title="Percent of respondents",
    xaxis2_title="Percent of respondents"
)

# make axes symmetric so left/right compare nicely
max_val = max(
    abs(min(x_pak + x_ong)),
    abs(max(x_pak + x_ong))
)
fig.update_xaxes(range=[-max_val, max_val], row=1, col=1)
fig.update_xaxes(range=[-max_val, max_val], row=1, col=2)

fig.show()


In [ ]:
columns_to_plot = [
    "What do you think about the number of dogs on your soi?",
]
for col in columns_to_plot:
    for i, (name, df) in enumerate([("Pak Phli", df1_pak), ("Ongkharak", df1_ong)], start=1):
        print('---')
        print(name)
        print(df[col].value_counts())

In [ ]:
for name, df in [("Pak Phli", df1_pak), ("Ongkharak", df1_ong)]:
    df_opinion = pd.DataFrame({
        "Negative": [(df.iloc[:, 86].isin(['I am unhappy', 'I do not accept'])).sum()],
        "Neutral": [(df.iloc[:, 86].isin(['Neutral'])).sum()],
        "Positive": [(df.iloc[:, 86].isin(['I am happy', 'I am accepting/OK'])).sum()],
    })

    print(f"{name}:\n{df_opinion}\n")

    fig = px.bar(
        x=df_opinion.columns,
        y=df_opinion.iloc[0],
        labels={'x': 'Opinion', 'y': 'Count'},
        title=f"{name} - Opinion on Number of Dogs",
        color=df_opinion.columns,
        color_discrete_map={
            "Negative": "#f64c4c",
            "Neutral": "#FACA63",
            "Positive": "#00CC96"
        }
    )
    fig.update_layout(width=1000, height=500)
    fig.show()


In [ ]:
print(f"{name}:\n{df_opinion}\n")


### Q55 – Over the last 12 months, has the number of dogs on your soi increased/decreased/stayed the same? 

In [ ]:
# Over the last 12 months, do you think there has been a change in the number of roaming dogs on your soi?


columns_to_plot = [
    "Over the last 12 months, do you think there has been a change in the number of roaming dogs on your soi?",
]

for col in columns_to_plot:
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Pak Phli", "Ongkharak"))
    for i, (name, df) in enumerate([("Pak Phli", df1_pak), ("Ongkharak", df1_ong)], start=1):
        counts = df[col].value_counts().sort_index()
        fig.add_trace(
            go.Bar(y=counts.values, x=counts.index, name=name),
            row=1, col=i
        )
    fig.update_layout(
        height=600,
        width=1100,
        title_text=col,
        xaxis_title="Number of Dogs",
        yaxis_title="Count",
        xaxis2_title="Number of Dogs",
        yaxis2_title="Count"
    )
    fig.show()

In [ ]:
ordered_categories = ['Decreased', 'No change', 'Increased']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=ordered_categories,
    y=[counts_pak.get(cat, 0) for cat in ordered_categories],
    name="Pak Phli"
))
fig.add_trace(go.Bar(
    x=ordered_categories,
    y=[counts_ong.get(cat, 0) for cat in ordered_categories],
    name="Ongkharak"
))

fig.update_layout(
    barmode='stack',
    title=col,
    xaxis_title="Change in Number of Dogs",
    yaxis_title="Count",
    width=1200,
    height=600
)
fig.show()


### Q56 – Compared to before COVID, what do you think about the number of dogs on your soi today? 

In [ ]:
columns_to_plot = [
    "Thinking of the number of dogs on your soi today vs. the first COVID lockdowns (2020), which statement is most true?",
]

for col in columns_to_plot:
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Pak Phli", "Ongkharak"))
    for i, (name, df) in enumerate([("Pak Phli", df1_pak), ("Ongkharak", df1_ong)], start=1):
        counts = df[col].value_counts().sort_index()
        fig.add_trace(
            go.Bar(y=counts.values, x=counts.index, name=name),
            row=1, col=i
        )
    fig.update_layout(
        height=600,
        width=1800,
        title_text=col,
        xaxis_title="Number of Dogs",   
        yaxis_title="Count",
        xaxis2_title="Number of Dogs",
        yaxis2_title="Count"
    )
    fig.show()

In [ ]:
columns_to_plot = [
    "Thinking of the number of dogs on your soi today vs. the first COVID lockdowns (2020), which statement is most true?",
]

for col in columns_to_plot:
    for i, (name, df) in enumerate([("Pak Phli", df1_pak), ("Ongkharak", df1_ong)], start=1):
        print(f'{name}\n{df[col].value_counts().sort_index()}\n')

In [ ]:
columns_to_plot = [
    "Thinking of the number of dogs on your soi today vs. the first COVID lockdowns (2020), which statement is most true?",
]

for col in columns_to_plot:
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Pak Phli", "Ongkharak"))
    for i, (name, df) in enumerate([("Pak Phli", df1_pak), ("Ongkharak", df1_ong)], start=1):
        counts = df[col].value_counts().drop("I didn’t live here at the time", errors="ignore")  # 🔑 drop category
        counts = counts.sort_index()
        fig.add_trace(
            go.Bar(y=counts.values, x=counts.index, name=name),
            row=1, col=i
        )
    fig.update_layout(
        height=600,
        width=1200,
        title_text=col,
        xaxis_title="Response",   
        yaxis_title="Count",
        xaxis2_title="Response",
        yaxis2_title="Count"
    )
    fig.show()


### Q57 – Over the last 12 months, do you think there has been a change in the level of trouble caused by dogs on your soi?

In [ ]:
columns_to_plot = [
    "Over the last 12 months, do you think there has been a change in how frequently you have been troubled by roaming dogs?",
]

for col in columns_to_plot:
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Pak Phli", "Ongkharak"))
    for i, (name, df) in enumerate([("Pak Phli", df1_pak), ("Ongkharak", df1_ong)], start=1):
        counts = df[col].value_counts().sort_index()
        fig.add_trace(
            go.Bar(y=counts.values, x=counts.index, name=name),
            row=1, col=i
        )
    fig.update_layout(
        height=600,
        width=1200,
        title_text=col,
        xaxis_title="Number of Dogs",   
        yaxis_title="Count",
        xaxis2_title="Number of Dogs",
        yaxis2_title="Count"
    )
    fig.show()

df.columns[89]

In [ ]:
columns_to_plot = [
    "Over the last 12 months, do you think there has been a change in how frequently you have been troubled by roaming dogs?",
]

for col in columns_to_plot:
    for i, (name, df) in enumerate([("Pak Phli", df1_pak), ("Ongkharak", df1_ong)], start=1):
        print(f'\n{name}\n{df[col].value_counts().sort_index()}')


In [ ]:
col = "Over the last 12 months, do you think there has been a change in how frequently you have been troubled by roaming dogs?"

color_map = {
    "Decreased": "#f64c4c",  # Negative
    "No change": "#FACA63",  # Neutral
    "Increased": "#00CC96"   # Positive
}

# Combine counts for both districts
counts = df1_filtered.groupby("District")[col].value_counts().unstack(fill_value=0)

# Plot stacked bar
fig = go.Figure()

for response in counts.columns:  # each response is a stack
    fig.add_trace(go.Bar(
        x=counts.index,  # districts
        y=counts[response], 
        name=response,
        marker_color=color_map.get(response, None)
    ))

fig.update_layout(
    barmode="stack",   # 🔑 stacked bars
    height=600,
    width=1200,
    title=col,
    xaxis_title="District",
    yaxis_title="Number of Respondents"
)

fig.show()


In [ ]:
col = "Over the last 12 months, do you think there has been a change in how frequently you have been troubled by roaming dogs?"

# Get counts grouped by category + district
counts = df1_filtered.groupby([col, "District"]).size().unstack(fill_value=0)

# Make stacked bar chart
fig = go.Figure()

for district in counts.columns:  # stack by district
    fig.add_trace(go.Bar(
        x=counts.index,   # categories: Increased / No change / Decreased
        y=counts[district],
        name=district
    ))

fig.update_layout(
    barmode="stack",   # 🔑 stacked
    height=600,
    width=1200,
    title=col,
    xaxis_title="Response",
    yaxis_title="Number of Respondents"
)

fig.show()


### Q58 – How should the roaming dog population in general be managed?

In [ ]:
df.columns[90]

In [ ]:
columns_to_plot = [
    "How should the roaming dog population in general be managed?",
]

for col in columns_to_plot:
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Pak Phli", "Ongkharak"))
    for i, (name, df) in enumerate([("Pak Phli", df1_pak), ("Ongkharak", df1_ong)], start=1):
        counts = df[col].value_counts().sort_index()
        fig.add_trace(
            go.Bar(y=counts.values, x=counts.index, name=name),
            row=1, col=i
        )
    fig.update_layout(
        height=600,
        width=1200,
        title_text=col,
        xaxis_title="Opinions",   
        yaxis_title="Count",
        xaxis2_title="Opinions",
        yaxis2_title="Count"
    )
    fig.show()

df.columns[89]